# D2L Reproduction Without a Pre-trained Generator

Test whether latent dreamed classes improve forward transfer when dreams are created only from the continual classifier. The experiment uses 50 Mini-ImageNet base classes followed by five 10-class tasks.

In [1]:
from dataclasses import dataclass
from pathlib import Path
import random

import numpy as np
import torch
from torch import Tensor, nn
import torch.nn.functional as F


@dataclass(frozen=True)
class Config:
    data_root: Path = Path("data/mini-imagenet")
    seed: int = 42
    base_classes: int = 50
    classes_per_task: int = 10
    num_tasks: int = 6
    bottleneck_dim: int = 256
    replay_size: int = 2_000
    dream_weight: float = 1.0


cfg = Config()
random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)

## 1. Build the task stream

Load Mini-ImageNet using a fixed class order. Task 0 contains 50 classes; tasks 1-5 contain 10 unseen classes each. Return separate train and test loaders and preserve the original labels.

In [ ]:
def build_task_stream(cfg: Config):
    """Return [(train_loader, test_loader, class_ids), ...] for all tasks."""
    # TODO: load Mini-ImageNet and apply the standard train/test transforms.
    # TODO: shuffle class IDs once with cfg.seed, then create the 50/10/... split.
    # TODO: use class-aware samplers without remapping labels between tasks.
    raise NotImplementedError


task_stream = build_task_stream(cfg)

## 2. Define the bottleneck classifier

Separate feature extraction from classification: $z=h_\phi(x)$ and $p(y\mid x)=\mathrm{softmax}(Wz+b)$. Allocate 100 output neurons from the start so unused future-class neurons can temporarily represent dreams.

In [ ]:
class BottleneckClassifier(nn.Module):
    def __init__(self, encoder: nn.Module, encoder_dim: int, cfg: Config):
        super().__init__()
        self.encoder = encoder
        self.bottleneck = nn.Linear(encoder_dim, cfg.bottleneck_dim)
        self.head = nn.Linear(cfg.bottleneck_dim, 100)

    def encode(self, x: Tensor) -> Tensor:
        return F.normalize(self.bottleneck(self.encoder(x)), dim=-1)

    def classify_features(self, z: Tensor) -> Tensor:
        return self.head(z)

    def forward(self, x: Tensor) -> Tensor:
        return self.classify_features(self.encode(x))


# TODO: instantiate a randomly initialized encoder; do not use ImageNet weights.
model: BottleneckClassifier

## 3. Train the 50-class base task

Train only on real base-task images using cross-entropy, while filling a reservoir replay buffer. No dreamed examples are stored in the buffer.

$$\mathcal L_{base}=-\mathbb E_{(x,y)\sim\mathcal D_0}\log p_\theta(y\mid x).$$

In [ ]:
class ReservoirBuffer:
    def __init__(self, capacity: int):
        self.capacity = capacity
        # TODO: store examples, labels, and the number of samples observed.

    def add_batch(self, images: Tensor, labels: Tensor) -> None:
        # TODO: implement reservoir sampling using real examples only.
        raise NotImplementedError

    def sample(self, batch_size: int) -> tuple[Tensor, Tensor]:
        raise NotImplementedError


def train_base_task(model, train_loader, buffer, optimizer, device):
    # TODO: optimize real-image cross-entropy over output slots 0-49.
    # TODO: update the replay buffer after each batch.
    raise NotImplementedError

## 4. Represent dreamed classes in the bottleneck

Replace D2L's diffusion prompt with a learned offset $\delta_k$. For real feature $z$, dream $k$ produces $z_k^d=\mathrm{norm}(z+\delta_k)$. Applying one offset to many real features gives a coherent dream distribution rather than a single prototype.

In [ ]:
class DreamBank(nn.Module):
    def __init__(self, slots: list[int], bottleneck_dim: int):
        super().__init__()
        self.slots = slots
        self.offsets = nn.Parameter(torch.zeros(len(slots), bottleneck_dim))

    def transform(self, z: Tensor) -> Tensor:
        # Shape: [num_dreams, batch, bottleneck_dim].
        return F.normalize(z.unsqueeze(0) + self.offsets.unsqueeze(1), dim=-1)


# After base training, slots 50-99 are available as dreamed classes.
dream_bank = DreamBank(list(range(50, 100)), cfg.bottleneck_dim)

## 5. Optimize structured dreams

Freeze the classifier and optimize only dream offsets. Dreams should be uncertain over seen real classes, remain a controlled distance from real features, form compact clusters, and stay separated from other dreams.

$$\mathcal L_{dream}=\lambda_u\mathrm{KL}(q_{real}(\cdot\mid z^d)\|U)+\lambda_s\mathcal L_{shell}+\lambda_c\mathcal L_{compact}+\lambda_d\mathcal L_{separate}.$$

In [ ]:
def dream_loss(
    model: BottleneckClassifier,
    dreamed: Tensor,
    real_features: Tensor,
    seen_slots: list[int],
) -> Tensor:
    # TODO: KL divergence from the uniform distribution over seen real slots.
    # TODO: shell loss to prevent arbitrary, far-away adversarial features.
    # TODO: within-dream compactness and between-dream margin separation.
    # TODO: weight and return the four terms; log each term separately.
    raise NotImplementedError


def optimize_dreams(model, dream_bank, real_loader, seen_slots, device):
    # TODO: freeze model parameters and optimize dream_bank.offsets only.
    # TODO: early-stop if entropy stalls or within-dream variance collapses.
    raise NotImplementedError

## 6. Map incoming classes onto dream slots

Before training a new task, measure how strongly each dreamed output responds to each incoming class. Use one-to-one maximum-score matching, then relabel the selected dream slots as the new real classes.

$$k^*(c)=\arg\max_{k\in\mathcal C_{dream}}\mathbb E_{x\sim\mathcal D_t^c}p_\theta(k\mid x).$$

In [ ]:
def map_classes_to_dreams(
    model: BottleneckClassifier,
    train_loader,
    class_ids: list[int],
    dream_slots: list[int],
    device,
) -> dict[int, int]:
    # TODO: compute a [num_new_classes, num_dreams] mean-probability matrix.
    # TODO: use Hungarian matching so each class receives a unique slot.
    # Return {original_class_id: output_slot}.
    raise NotImplementedError

## 7. Train each incremental task

Train jointly on current real images, replayed real images, and residual dreamed features. Keep dream transformations connected to $h_\phi(x)$ so the dream loss can update the encoder, not only the final head.

$$\mathcal L_t=\mathcal L_{current}+\lambda_r\mathcal L_{replay}+\lambda_d\mathcal L_{dream\_CE}.$$

In [ ]:
def train_incremental_task(
    model,
    dream_bank,
    train_loader,
    class_to_slot,
    buffer,
    optimizer,
    device,
):
    # TODO: remap current labels to their selected output slots.
    # TODO: compute current and replay cross-entropy losses.
    # TODO: transform current/replay features with each residual dream offset.
    # TODO: classify those features using their dream slots and update the model.
    raise NotImplementedError

## 8. Run the continual-learning cycle

For each incremental task: evaluate before learning, map classes to dreams, retire the claimed dream slots, train, update the real replay buffer, then re-optimize the remaining dream inventory. Do not dream after the final task.

In [ ]:
def run_experiment(cfg: Config):
    # TODO: initialize loaders, model, optimizer, buffer, and metric matrix.
    # TODO: train task 0, then optimize the initial dream inventory.
    # TODO: execute map -> train -> buffer update -> dream refresh for tasks 1-5.
    # TODO: save checkpoints, mappings, dream statistics, and task accuracies.
    raise NotImplementedError


# results = run_experiment(cfg)

## 9. Evaluate the hypothesis

Report average accuracy, forgetting, and forward transfer. Compare rehearsal-only, random latent dreams, uncertainty-only dreams, structured dreams without slot reuse, and the full method.

$$\mathrm{FWT}=\frac{1}{T-1}\sum_{t=1}^{T-1}(A_{t-1,t}-B_t).$$

In [ ]:
def compute_metrics(accuracy_matrix: np.ndarray, baselines: np.ndarray) -> dict[str, float]:
    # TODO: compute final average accuracy, average forgetting, and FWT.
    raise NotImplementedError


ABLATIONS = [
    "replay_only",
    "random_dreams",
    "uncertainty_only",
    "structured_no_reuse",
    "structured_with_reuse",
]